# - Ahmedabad Real Estate Analysis 2026 -

# Live property listings scraped from MagicBricks across 10 Ahmedabad Diverse localities to identify best value areas, market activity, and dominant buyer profiles covering the full analyst workflow
  
**Data:** From MagicBricks · May 2026  
**Tools:** Python · BeautifulSoup · Pandas · MySQL · Power BI

# Phase 1 - Data Scrapping

In [72]:

import mysql.connector
import pandas as pd
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re

print("All imports done ✓")

All imports done ✓


In [59]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}

locality_urls = {
    "Prahlad Nagar": "https://www.99acres.com/search/property/buy/flat-in-prahlad-nagar?city=13&preference=S&res_com=R",
    "Bopal":         "https://www.99acres.com/search/property/buy/flat-in-bopal?city=13&preference=S&res_com=R",
    "Satellite":     "https://www.99acres.com/search/property/buy/flat-in-satellite?city=13&preference=S&res_com=R",
    "SBR":           "https://www.99acres.com/search/property/buy/flat-in-sindhu-bhavan-road?city=13&preference=S&res_com=R",
    "Thaltej":       "https://www.99acres.com/search/property/buy/flat-in-thaltej?city=13&preference=S&res_com=R",
    "Naranpura":     "https://www.99acres.com/search/property/buy/flat-in-naranpura?city=13&preference=S&res_com=R",
    "Chandkheda":    "https://www.99acres.com/search/property/buy/flat-in-chandkheda?city=13&preference=S&res_com=R",
    "Nikol":         "https://www.99acres.com/search/property/buy/flat-in-nikol?city=13&preference=S&res_com=R",
    "Khodiyar Nagar":"https://www.99acres.com/search/property/buy/flat-in-khodiyar-nagar?city=13&preference=S&res_com=R",
    "Science City":  "https://www.99acres.com/search/property/buy/flat-in-science-city-road?city=13&preference=S&res_com=R",
}

print(f"{len(locality_urls)} localities loaded ✓")

10 localities loaded ✓


In [60]:
test_url = locality_urls["Bopal"] + "&page=1"
response = requests.get(test_url, headers=headers)
print("Status code:", response.status_code)

if response.status_code == 200:
    print("✓ Connected to MagicBricks successfully")
else:
    print("✗ Something went wrong — paste the status code here")

Status code: 200
✓ Connected to MagicBricks successfully


In [61]:
soup = BeautifulSoup(response.text, "html.parser")

cards = soup.find_all("div", class_="mb-srp__card")
print(f"Cards found: {len(cards)}")

Cards found: 30


In [62]:
card = cards[0]

# Title — contains BHK + locality both
title = card.find("h2", class_="mb-srp__card--title")
title_text = title.text.strip() if title else "NOT FOUND"

# Area
area = card.find("div", class_="mb-srp__card__summary--value")
area_text = area.text.strip() if area else "NOT FOUND"

# Price
price = card.find("div", class_="mb-srp__card__price--amount")
price_text = price.text.strip() if price else "NOT FOUND"

# Price per sqft — already calculated by MagicBricks
ppsf = card.find("div", class_="mb-srp__card__price--size")
ppsf_text = ppsf.text.strip() if ppsf else "NOT FOUND"

print("Title:          ", title_text)
print("Area:           ", area_text)
print("Price:          ", price_text)
print("Price per sqft: ", ppsf_text)

Title:           2 BHK Flat for Sale in Mahadev Glory, Bopal, Ahmedabad
Area:            1350 sqft
Price:           ₹65 Lac
Price per sqft:  ₹4815 per sqft


In [63]:
all_data = []
PAGES = 5

for locality_name, base_url in locality_urls.items():
    print(f"\nScraping: {locality_name}")
    count = 0

    for page in range(1, PAGES + 1):
        url = base_url + f"&page={page}"

        try:
            resp = requests.get(url, headers=headers, timeout=15)

            if resp.status_code != 200:
                print(f"  Page {page}: blocked ({resp.status_code}) — skipping")
                time.sleep(10)
                continue

            soup  = BeautifulSoup(resp.text, "html.parser")
            cards = soup.find_all("div", class_="mb-srp__card")

            if len(cards) == 0:
                print(f"  Page {page}: no listings — stopping this locality")
                break

            for card in cards:
                try:
                    title_tag = card.find("h2",  class_="mb-srp__card--title")
                    area_tag  = card.find("div", class_="mb-srp__card__summary--value")
                    price_tag = card.find("div", class_="mb-srp__card__price--amount")
                    ppsf_tag  = card.find("div", class_="mb-srp__card__price--size")

                    title_text = title_tag.text.strip() if title_tag else ""

                    price_lakh = clean_price(price_tag.text  if price_tag else "")
                    area_sqft  = clean_area(area_tag.text    if area_tag  else "")
                    ppsf       = clean_ppsf(ppsf_tag.text    if ppsf_tag  else "")
                    bhk        = extract_bhk(title_text)
                    locality_detail = extract_locality(title_text, fallback_locality=locality_name)

                    if price_lakh and area_sqft and ppsf:
                        all_data.append({
                            "locality":        locality_name,
                            "locality_detail": locality_detail,
                            "bhk":             bhk,
                            "price_lakh":      price_lakh,
                            "area_sqft":       area_sqft,
                            "price_per_sqft":  ppsf,
                        })
                        count += 1

                except:
                    pass

            print(f"  Page {page}: ✓  ({count} listings so far)")
            time.sleep(3)

        except Exception as e:
            print(f"  Page {page}: error — {e}")
            time.sleep(10)

    print(f"  Done — {count} listings from {locality_name}")

print(f"\n✓ Scraping complete — {len(all_data)} total listings")


Scraping: Prahlad Nagar
  Page 1: ✓  (30 listings so far)
  Page 2: ✓  (60 listings so far)
  Page 3: ✓  (88 listings so far)
  Page 4: ✓  (117 listings so far)
  Page 5: ✓  (146 listings so far)
  Done — 146 listings from Prahlad Nagar

Scraping: Bopal
  Page 1: ✓  (30 listings so far)
  Page 2: ✓  (60 listings so far)
  Page 3: ✓  (90 listings so far)
  Page 4: ✓  (119 listings so far)
  Page 5: ✓  (148 listings so far)
  Done — 148 listings from Bopal

Scraping: Satellite
  Page 1: ✓  (29 listings so far)
  Page 2: ✓  (58 listings so far)
  Page 3: ✓  (85 listings so far)
  Page 4: ✓  (111 listings so far)
  Page 5: ✓  (137 listings so far)
  Done — 137 listings from Satellite

Scraping: SBR
  Page 1: ✓  (25 listings so far)
  Page 2: ✓  (54 listings so far)
  Page 3: ✓  (84 listings so far)
  Page 4: ✓  (114 listings so far)
  Page 5: ✓  (144 listings so far)
  Done — 144 listings from SBR

Scraping: Thaltej
  Page 1: ✓  (27 listings so far)
  Page 2: ✓  (57 listings so far)
  Pag

## Phase 2 - Cleaning

In [64]:
df = pd.DataFrame(all_data)

# Remove unrealistic values
df = df[df["price_per_sqft"] > 1500]
df = df[df["price_per_sqft"] < 25000]
df = df[df["area_sqft"]      > 200  ]
df = df.dropna(subset=["price_lakh", "area_sqft", "price_per_sqft"])

df.to_csv("ahmedabad_clean.csv", index=False)
print(f"✓ Saved — {len(df)} clean listings")
print(df.head())

✓ Saved — 1138 clean listings
        locality locality_detail  bhk  price_lakh  area_sqft  price_per_sqft
0  Prahlad Nagar   Prahlad Nagar  4.0       271.0     2075.0          7200.0
1  Prahlad Nagar   Prahlad Nagar  4.0       271.0     1940.0          7200.0
2  Prahlad Nagar   Prahlad Nagar  4.0       271.0     1828.0          7200.0
3  Prahlad Nagar   Prahlad Nagar  4.0       400.0     4015.0          9963.0
4  Prahlad Nagar   Prahlad Nagar  4.0       285.0     2651.0         10751.0


In [65]:
print("=" * 55)
print("Q1: BEST VALUE — avg price per sqft by locality")
print("=" * 55)
q1 = df.groupby("locality")["price_per_sqft"].mean().round(0).sort_values()
print(q1.to_string())

print("\n")
print("=" * 55)
print("Q2: MARKET ACTIVITY — listing count by locality")
print("=" * 55)
q2 = df.groupby("locality").size().sort_values(ascending=False)
print(q2.to_string())

print("\n")
print("=" * 55)
print("Q3: DOMINANT BHK TYPE per locality")
print("=" * 55)
q3 = df.groupby(["locality","bhk"]).size().unstack(fill_value=0)
print(q3.to_string())

print("\n")
cheapest  = q1.index[0]
costliest = q1.index[-1]
premium   = round((q1.iloc[-1] - q1.iloc[0]) / q1.iloc[0] * 100, 1)
print("=" * 55)
print("YOUR RESUME LINE:")
print("=" * 55)
print(f"{costliest} is {premium}% more expensive per sqft than {cheapest}")

Q1: BEST VALUE — avg price per sqft by locality
locality
Nikol             4754.0
Chandkheda        5028.0
Khodiyar Nagar    5402.0
Bopal             5771.0
SBR               6274.0
Science City      6669.0
Naranpura         6794.0
Prahlad Nagar     7891.0
Thaltej           7955.0
Satellite         8498.0


Q2: MARKET ACTIVITY — listing count by locality
locality
Bopal             147
Prahlad Nagar     145
SBR               137
Thaltej           137
Naranpura         130
Science City      128
Satellite         118
Chandkheda        105
Nikol              72
Khodiyar Nagar     19


Q3: DOMINANT BHK TYPE per locality
bhk             1.0  2.0  3.0  4.0  5.0  6.0  7.0
locality                                         
Bopal             1   26   83   35    2    0    0
Chandkheda        5   43   49    7    0    0    0
Khodiyar Nagar    0    5   10    4    0    0    0
Naranpura         1    9   99   20    1    0    0
Nikol            15   31   20    6    0    0    0
Prahlad Nagar     2   22   

In [66]:
df_raw = pd.DataFrame(all_data)
df_raw.to_csv("ahmedabad.realestate2026_raw.csv", index=False)
print("Saved!")

Saved!


In [67]:
import mysql.connector
import pandas as pd

# Load clean CSV with full path
df = pd.read_csv("/Users/aryan/Desktop/Data Analytics/Ahmedabad Real estate 2026/ahmedabad_clean.csv")

# Rename columns to match SQL table
df.columns = ["locality", "locality_detail", "bhk", "price_lakh", "area_sqft", "price_per_sqft"]

# Connect to MySQL
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="9727267345"
)
cursor = conn.cursor()

# Create database and table
cursor.execute("CREATE DATABASE IF NOT EXISTS ahmedabad_realestate")
cursor.execute("USE ahmedabad_realestate")
cursor.execute("""
    CREATE TABLE IF NOT EXISTS ahmedabad_listings (
        id              INT AUTO_INCREMENT PRIMARY KEY,
        locality        VARCHAR(100),
        locality_detail VARCHAR(200),
        bhk             INT,
        price_lakh      DECIMAL(10,2),
        area_sqft       DECIMAL(10,2),
        price_per_sqft  DECIMAL(10,2)
    )
""")

# Insert all rows
inserted = 0
for _, row in df.iterrows():
    cursor.execute("""
        INSERT INTO ahmedabad_listings
        (locality, locality_detail, bhk, price_lakh, area_sqft, price_per_sqft)
        VALUES (%s, %s, %s, %s, %s, %s)
    """, (
        row["locality"],
        row["locality_detail"],
        int(row["bhk"])              if pd.notna(row["bhk"])              else None,
        float(row["price_lakh"])     if pd.notna(row["price_lakh"])       else None,
        float(row["area_sqft"])      if pd.notna(row["area_sqft"])        else None,
        float(row["price_per_sqft"]) if pd.notna(row["price_per_sqft"])   else None,
    ))
    inserted += 1

conn.commit()
print(f"✓ {inserted} rows inserted into MySQL successfully")

cursor.close()
conn.close()



✓ 1105 rows inserted into MySQL successfully


In [68]:
df = pd.read_csv("/Users/aryan/Desktop/Data Analytics/Ahmedabad Real estate 2026/ahmedabad_clean.csv")

print("Before:", len(df), "rows")

# Remove unrealistic prices
df = df[df["price_lakh"] >= 15]      # no flat under ₹15L in Ahmedabad
df = df[df["price_lakh"] <= 500]     # no flat over ₹5Cr in this dataset
df = df[df["price_per_sqft"] >= 2000] # below ₹2000/sqft is unrealistic
df = df[df["price_per_sqft"] <= 20000] # above ₹20000/sqft is outlier
df = df[df["bhk"] <= 5]              # remove 6BHK and 7BHK outliers

print("After:", len(df), "rows")

df.to_csv("/Users/aryan/Desktop/Data Analytics/Ahmedabad Real estate 2026/ahmedabad_clean.csv", index=False)
print("Saved ✓")

Before: 1105 rows
After: 1105 rows
Saved ✓


In [69]:
df = pd.read_csv("/Users/aryan/Desktop/Data Analytics/Ahmedabad Real estate 2026/ahmedabad_clean.csv")

print("Total rows:", len(df))
print("Max price_lakh:", df["price_lakh"].max())
print("Min price_lakh:", df["price_lakh"].min())
print("Max price_per_sqft:", df["price_per_sqft"].max())
print("Max BHK:", df["bhk"].max())

Total rows: 1105
Max price_lakh: 500.0
Min price_lakh: 16.0
Max price_per_sqft: 16988.0
Max BHK: 5.0


## Phase 3 - SQL Analysis

In [70]:
import mysql.connector
import pandas as pd

# ============================================================
# PHASE 4 — SQL ANALYSIS
# Project  : Ahmedabad Real Estate Price Intelligence (2026)
# Data     : 1,105 listings scraped from MagicBricks
# Tool     : MySQL via Python connector
# Author   : Aryansinh Chauhan
# ============================================================

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="9727267345"
)
cursor = conn.cursor()
cursor.execute("USE ahmedabad_realestate")

print("Connected to MySQL ✓")
print("Database : ahmedabad_realestate")
print("Table    : ahmedabad_listings")
print("Rows     : 1,105 live property listings")


# ============================================================
# QUERY 1 — BEST VALUE LOCALITY
# ============================================================
# Question : Which locality gives the most space for money?
# Method   : Average price per sqft grouped by locality
# Use case : Helps budget buyers identify where to look first
# ============================================================

print("\n")
print("=" * 55)
print("QUERY 1 — Best Value Locality (Cheapest → Costliest)")
print("=" * 55)

q1 = """
    SELECT   locality,
             ROUND(AVG(price_per_sqft), 0) AS avg_price_sqft
    FROM     ahmedabad_listings
    GROUP BY locality
    ORDER BY avg_price_sqft ASC
"""

df_q1 = pd.read_sql(q1, conn)
df_q1.index = range(1, len(df_q1) + 1)
print(df_q1.to_string())

cheapest  = df_q1.iloc[0]["locality"]
costliest = df_q1.iloc[-1]["locality"]
cheap_val = df_q1.iloc[0]["avg_price_sqft"]
cost_val  = df_q1.iloc[-1]["avg_price_sqft"]
premium   = round((cost_val - cheap_val) / cheap_val * 100, 1)

print()
print(f"INSIGHT → {costliest} is {premium}% more expensive per sqft")
print(f"          than {cheapest} — widest price gap in the dataset.")


# ============================================================
# QUERY 2 — MARKET ACTIVITY
# ============================================================
# Question : Which localities have the most listings?
# Method   : COUNT listings grouped by locality
# Use case : More listings = more buyer options = better
#            resale potential and easier price comparison
# ============================================================

print("\n")
print("=" * 55)
print("QUERY 2 — Market Activity (Most → Least Active)")
print("=" * 55)

q2 = """
    SELECT   locality,
             COUNT(*) AS listing_count
    FROM     ahmedabad_listings
    GROUP BY locality
    ORDER BY listing_count DESC
"""

df_q2 = pd.read_sql(q2, conn)
df_q2.index = range(1, len(df_q2) + 1)
print(df_q2.to_string())

most_active  = df_q2.iloc[0]["locality"]
least_active = df_q2.iloc[-1]["locality"]
most_count   = int(df_q2.iloc[0]["listing_count"])
least_count  = int(df_q2.iloc[-1]["listing_count"])

print()
print(f"INSIGHT → {most_active} is the most liquid market")
print(f"          with {most_count} listings — {round(most_count/least_count,1)}x more active")
print(f"          than {least_active} ({least_count} listings).")


# ============================================================
# QUERY 3 — DOMINANT BHK TYPE PER LOCALITY
# ============================================================
# Question : What flat type is most common in each area?
# Method   : COUNT grouped by locality + bhk, pick top 1
# Use case : Reveals buyer profile per locality
#            4BHK dominant = luxury/premium buyers
#            3BHK dominant = family segment buyers
#            2BHK dominant = middle income/first time buyers
# ============================================================

print("\n")
print("=" * 55)
print("QUERY 3 — Dominant BHK Type per Locality")
print("=" * 55)

q3 = """
    SELECT   locality,
             bhk,
             COUNT(*) AS listing_count
    FROM     ahmedabad_listings
    GROUP BY locality, bhk
    ORDER BY locality, listing_count DESC
"""

df_q3       = pd.read_sql(q3, conn)
df_dominant = df_q3.groupby("locality").first().reset_index()
df_dominant.index = range(1, len(df_dominant) + 1)
df_dominant.columns = ["locality", "dominant_bhk", "listing_count"]
print(df_dominant.to_string())

print()
print("INSIGHT → SBR and Thaltej are dominated by 4BHK —")
print("          signalling a luxury buyer profile.")
print("          Nikol is the only 2BHK-dominant locality —")
print("          indicating a first-time/middle-income market.")


# ============================================================
# QUERY 4 — BEST 3BHK UNDER ₹80 LAKH
# ============================================================
# Question : Where is the best value 3BHK for a family
#            with a budget of ₹80 lakh or less?
# Method   : Filter bhk=3 and price_lakh<=80, then rank
#            by avg price per sqft ascending
# Use case : Most practical query for a real buyer —
#            3BHK under ₹80L is the Ahmedabad family sweet spot
# ============================================================

print("\n")
print("=" * 55)
print("QUERY 4 — Best 3BHK Flats Under ₹80 Lakh")
print("=" * 55)

q4 = """
    SELECT   locality,
             ROUND(AVG(price_per_sqft), 0) AS avg_price_sqft,
             ROUND(AVG(area_sqft), 0)      AS avg_area_sqft,
             ROUND(AVG(price_lakh), 1)     AS avg_price_lakh,
             COUNT(*)                       AS listings_available
    FROM     ahmedabad_listings
    WHERE    bhk = 3
    AND      price_lakh <= 80
    GROUP BY locality
    ORDER BY avg_price_sqft ASC
"""

df_q4 = pd.read_sql(q4, conn)
df_q4.index = range(1, len(df_q4) + 1)
print(df_q4.to_string())

if len(df_q4) > 0:
    best = df_q4.iloc[0]
    print()
    print(f"INSIGHT → {best['locality']} offers the best value 3BHK")
    print(f"          under ₹80L at ₹{int(best['avg_price_sqft'])}/sqft")
    print(f"          avg area {int(best['avg_area_sqft'])} sqft across")
    print(f"          {int(best['listings_available'])} available listings.")



Connected to MySQL ✓
Database : ahmedabad_realestate
Table    : ahmedabad_listings
Rows     : 1,105 live property listings


QUERY 1 — Best Value Locality (Cheapest → Costliest)
          locality  avg_price_sqft
1            Nikol          4706.0
2   Khodiyar Nagar          4912.0
3       Chandkheda          4987.0
4            Bopal          5687.0
5              SBR          6149.0
6     Science City          6658.0
7        Naranpura          6878.0
8    Prahlad Nagar          7851.0
9          Thaltej          7970.0
10       Satellite          8519.0

INSIGHT → Satellite is 81.0% more expensive per sqft
          than Nikol — widest price gap in the dataset.


QUERY 2 — Market Activity (Most → Least Active)
          locality  listing_count
1            Bopal            734
2    Prahlad Nagar            720
3          Thaltej            679
4              SBR            674
5        Naranpura            650
6     Science City            624
7        Satellite            579
8    

/var/folders/m2/twcqy9gx6kg5stnkn4n162h80000gn/T/ipykernel_14900/1096045102.py:47: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_q1 = pd.read_sql(q1, conn)
/var/folders/m2/twcqy9gx6kg5stnkn4n162h80000gn/T/ipykernel_14900/1096045102.py:84: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_q2 = pd.read_sql(q2, conn)
/var/folders/m2/twcqy9gx6kg5stnkn4n162h80000gn/T/ipykernel_14900/1096045102.py:124: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_q3       = pd.read_sql(q3, conn)
/var/folders/m2/twcqy9gx6kg5stnkn4n162h80000g

## Phase 4 - Visualization On Microsoft Power Bi